In [ ]:
from google.colab import files
uploaded = files.upload()


Saving datasets_challenge.zip to datasets_challenge.zip
Saving models.zip to models.zip
Saving utils.zip to utils.zip


In [ ]:
!unzip -q utils.zip

In [ ]:
!unzip -q models.zip

In [ ]:
!unzip -q datasets_challenge.zip

In [ ]:
import os
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
import contextlib
import plotly.graph_objects as go

from utils.reproducibility_utils import seed_everything
# ✅ CRITICAL FIX: Use hybrid validation (creates random + temporal validation sets)
from utils.data_loadingV2 import load_data, load_test_data, feature_names
from utils.LDS_FocalR_Loss import EnhancedLDS_FocalR_Loss
from utils.SimpleBucketLoss import SimpleBucketLoss
from utils.kaggle_metric import score
from models.gru import GRUModel

plt.style.use('seaborn-v0_8')

In [ ]:
CONFIG = {
    # Paths
    'TRAIN_PATH': "datasets_challenge/train",
    'TEST_PATH': "datasets_challenge/test",
    'DIST_CSV': "datasets_challenge/sideSlip_distribution.csv",
    'CHECKPOINT_NAME': "best_model_FIXED_experiment1.pth",  # ✅ New name for fixed version
    'SUBMISSION_CSV': "predictions_FIXED_experiment1.csv",

    # Reproducibility
    'SEED': 42,

    # Data Loading - V2 with validation file
    'VAL_FILE': 'train_part_006.csv',
    'VAL_RATIO': 0.0,  # ✅ Set to 0.0 to use only file-based validation (for now)
    'BATCH_SIZE': 512,
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': "vxCG",
    'COLUMNS_TO_DROP': ["sideSlip", "vyCG", "time", "longitude", "latitude"],
    'TARGET': "sideSlip",
    'SCALING': True,

    # Model Architecture - GRU (NOW WITH FIXES!)
    'MODEL_TYPE': 'GRU_Fixed',
    'RNN_UNITS': 256,
    'FC_UNITS': 256,
    'NUM_LAYERS': 3,
    'DROPOUT': [0.0, 0.0, 0.0],  # ✅ CRITICAL FIX: Restored dropout (was [0.0, 0.0, 0.0])
    'USE_BATCH_NORM': True,  # LayerNorm is now used instead

    # Loss Configuration - FIXED WEIGHTS!
    'LOSS_TYPE': 'EnhancedLDS_FocalR_Loss',
    'LOSS_WEIGHTS': (1.0, 2.0, 4.0),  # ✅ CRITICAL FIX: Few-shot gets 4x weight (was 1.5, 1.2, 1.0)
    'USE_HUBER': True,
    'HUBER_BETA': 0.08,  # Back to 0.08 (more stable than 0.07)
    'LDS_SIGMA': 2.5,  # ✅ Slightly increased for better smoothing (was 2.0)
    'FOCAL_GAMMA': 1.0,
    'BINS': 100,
    'INCLUDE_VAL_IN_LDS': True,  # ✅ NEW: Include validation samples in bin statistics

    # Optimizer
    'LEARNING_RATE': 5e-4,
    'WEIGHT_DECAY': 5e-4,  # Back to 5e-4 (6e-4 was too aggressive)
    'BETAS': (0.9, 0.999),
    'MAX_GRAD_NORM': 1.0,

    # Scheduler
    'USE_SCHEDULER': True,
    'SCHEDULER_T_MAX': 400,
    'SCHEDULER_ETA_MIN': 1e-5,

    # Training
    'NUM_EPOCHS': 100,
    'EARLY_STOP_PATIENCE': 15,
    'USE_AMP': True,

    # Challenge Constraints
    'MANY_SHOT_CONSTRAINT': 0.2,       # Kaggle hard limit
    'MANY_SHOT_SAFETY_MARGIN': 0.19,   # Validation target (safety margin for test)
    'FEW_SHOT_TARGET': 1.0,  # Updated to realistic target

    # Visualization
    'FIG_SIZE': (14, 10),
    'PLOTLY_WIDTH': 1200,
    'PLOTLY_HEIGHT': 500,
}



In [ ]:
import glob
def list_train_files(train_path):
    return sorted([os.path.basename(f) for f in glob.glob(os.path.join(train_path, "*.csv"))])

In [ ]:
import random
def make_file_folds(files, k=5, seed=42, shuffle=True):
    files = sorted(files)
    if shuffle:
        random.seed(seed)
        random.shuffle(files)
    fold_size = len(files) // k
    folds = [files[i:i+1] for i in range(0, len(files), fold_size)]  # one file per fold
    return folds[:k]

In [ ]:
def band_mae_np(y_true, y_pred):
    a = np.abs(y_true)
    e = np.abs(y_true - y_pred)
    many = e[a <= 1.0].mean() if np.any(a <= 1.0) else np.nan
    med = e[(a > 1.0) & (a <= 5.0)].mean() if np.any((a > 1.0) & (a <= 5.0)) else np.nan
    few = e[a > 5.0].mean() if np.any(a > 5.0) else np.nan
    return many, med, few

In [ ]:
num_features = len(feature_names) - len(CONFIG["COLUMNS_TO_DROP"])

In [ ]:
seed_everything(CONFIG['SEED'])
print(f"Random seed set to {CONFIG['SEED']}")

Random seed set to 42


In [ ]:
import copy
def run_one_fold(CONFIG, feature_names, val_file):
    """
    Train a single fold using val_file as validation set.
    Returns: dict of fold metrics and checkpoint name.
    """
    print(f"\n🧩 Starting fold with validation file: {val_file}")
    cfg = copy.deepcopy(CONFIG)
    cfg['VAL_FILE'] = val_file
    cfg['CHECKPOINT_NAME'] = f"best_fold_{val_file.replace('.csv','')}.pth"

    # Load data
    train_dataloader, val_dataloader, pipeline, mean_dict, std_dict = load_data(
        train_path=cfg['TRAIN_PATH'],
        val_file=val_file,  # REQUIRED: Dedicated validation file
        columns_to_standardize=feature_names,
        columns_to_drop=cfg['COLUMNS_TO_DROP'],
        sequence_length=cfg['SEQUENCE_LENGTH'],
        threshold=cfg['THRESHOLD'],
        threshold_value=cfg['THRESHOLD_VALUE'],
        threshold_column=cfg['THRESHOLD_COLUMN'],
        target=cfg['TARGET'],
        scaling=cfg['SCALING'],
        seed=cfg['SEED'],
        shuffle=True
    )

    print(f"\nTraining batches: {len(train_dataloader)}")
    print(f"Validation batches: {len(val_dataloader)}")
    print(f"Train samples: {len(train_dataloader.dataset):,}")
    print(f"Val samples: {len(val_dataloader.dataset):,}")

    # Display standardization statistics
    print(f"\n📊 Standardization Statistics (from V2):")
    print(f"   Features standardized: {len(mean_dict)}")
    if mean_dict:
        print(f"   Example - vxCG: mean={mean_dict.get('vxCG', 'N/A'):.4f}, std={std_dict.get('vxCG', 'N/A'):.4f}")
        print(f"   Example - ayCG: mean={mean_dict.get('ayCG', 'N/A'):.4f}, std={std_dict.get('ayCG', 'N/A'):.4f}")


    # Extract training targets for loss function
    all_train_targets = []
    for i in range(len(train_dataloader.dataset)):
        _, y = train_dataloader.dataset[i]
        all_train_targets.append(y.item())

    # ✅ CRITICAL FIX: Include validation samples in LDS bin statistics
    # This ensures LDS weights cover the full distribution, especially rare few-shot samples
    if cfg['INCLUDE_VAL_IN_LDS']:
        print(f"\n✅ Including validation samples in LDS bin statistics...")
        all_val_targets = []
        for i in range(len(val_dataloader.dataset)):
            _, y = val_dataloader.dataset[i]
            all_val_targets.append(y.item())

        # Combine train + val for LDS statistics (but NOT for training!)
        y_train_np = np.array(all_train_targets + all_val_targets)
        print(f"   Training samples: {len(all_train_targets):,}")
        print(f"   Validation samples: {len(all_val_targets):,}")
        print(f"   Total for LDS: {len(y_train_np):,}")
    else:
        y_train_np = np.array(all_train_targets)
        print(f"Training samples (LDS): {len(y_train_np):,}")

    # Initialize model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # AMP setup
    if torch.cuda.is_available():
        amp_device_type = "cuda"
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        scaler = torch.amp.GradScaler(amp_device_type, enabled=CONFIG['USE_AMP'])
    else:
        amp_device_type = None
        amp_dtype = None
        scaler = torch.amp.GradScaler("cuda", enabled=False)

    def autocast_ctx():
        if amp_device_type is None or not CONFIG['USE_AMP']:
            return contextlib.nullcontext()
        return torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype, enabled=True)

    # Get input size
    X_sample, _ = next(iter(train_dataloader))
    input_size = X_sample.shape[2]
    seq_length = X_sample.shape[1]

    print(f"\n📊 Input shape: (batch={X_sample.shape[0]}, seq={seq_length}, features={input_size})")

    # Create GRU model
    model = GRUModel(
        input_size=input_size,
        output_size=1,
        rnn_units=CONFIG['RNN_UNITS'],
        fc_units=CONFIG['FC_UNITS'],
        dropout=CONFIG['DROPOUT'],
        num_layers=CONFIG['NUM_LAYERS'],
        use_batch_norm=CONFIG['USE_BATCH_NORM']
    ).to(device)

    # Create loss function
    if CONFIG['LOSS_TYPE'] == 'SimpleBucket':
        criterion = SimpleBucketLoss(
            train_targets=y_train_np,
            bucket_weights=CONFIG['LOSS_WEIGHTS'],
            use_huber=CONFIG['USE_HUBER'],
            huber_beta=CONFIG['HUBER_BETA'],
            dist_csv_path=CONFIG['DIST_CSV']
        ).to(device)
    else:
        criterion = EnhancedLDS_FocalR_Loss(
            train_targets=y_train_np,
            bucket_weights=CONFIG['LOSS_WEIGHTS'],
            lds_kernel='gaussian',
            lds_ks=CONFIG['LDS_SIGMA'],
            lds_sigma=CONFIG['LDS_SIGMA'],
            focal_gamma=CONFIG['FOCAL_GAMMA'],
            use_huber=CONFIG['USE_HUBER'],
            huber_beta=CONFIG['HUBER_BETA'],
            num_bins=CONFIG['BINS'],
            dist_csv_path=CONFIG['DIST_CSV']
        ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n🔧 Model: {CONFIG['MODEL_TYPE']}")
    print(f"   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")

    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG['LEARNING_RATE'],
        weight_decay=CONFIG['WEIGHT_DECAY'],
        betas=CONFIG['BETAS']
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG['SCHEDULER_T_MAX'],
        eta_min=CONFIG['SCHEDULER_ETA_MIN']
    ) if CONFIG['USE_SCHEDULER'] else None

    # Training state
    train_curve, val_curve = [], []
    train_mae_curve, val_mae_curve = [], []
    bucket_many, bucket_medium, bucket_few = [], [], []
    learning_rate_curve = []
    best_loss = float("inf")
    best_epoch = -1
    epochs_no_improve = 0

    print("Training setup complete")

    # Training loop
    for epoch in range(CONFIG['NUM_EPOCHS']):
    # Training
        model.train()
        running_loss, running_mae = 0.0, 0.0
        num_train_batches = len(train_dataloader)

        for xb, yb in train_dataloader:
            xb = xb.to(device, non_blocking=True).float()
            yb = yb.to(device, non_blocking=True).float().reshape(-1, 1)

            optimizer.zero_grad(set_to_none=True)

            with autocast_ctx():
                pred = model(xb)
                total_loss = criterion(pred, yb)

            scaler.scale(total_loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['MAX_GRAD_NORM'])
            scaler.step(optimizer)
            scaler.update()

            running_loss += total_loss.item()
            running_mae += (pred - yb).abs().mean().item()

        avg_train_loss = running_loss / num_train_batches
        avg_train_mae = running_mae / num_train_batches
        train_curve.append(avg_train_loss)
        train_mae_curve.append(avg_train_mae)

        # Validation
        model.eval()
        running_vloss, running_vmae = 0.0, 0.0
        all_val_preds, all_val_targets = [], []
        num_val_batches = len(val_dataloader)

        with torch.no_grad():
            for xv, yv in val_dataloader:
                xv = xv.to(device, non_blocking=True).float()
                yv = yv.to(device, non_blocking=True).float().reshape(-1, 1)

                with autocast_ctx():
                    pred_val = model(xv)
                    total_loss_val = criterion(pred_val, yv)

                running_vloss += total_loss_val.item()
                running_vmae += (pred_val - yv).abs().mean().item()
                all_val_preds.append(pred_val.cpu())
                all_val_targets.append(yv.cpu())

        all_val_preds = torch.cat(all_val_preds, dim=0).to(device)
        all_val_targets = torch.cat(all_val_targets, dim=0).to(device)
        best_buckets = criterion.bucket_metrics(all_val_preds, all_val_targets)
        best_vloss = running_vloss / num_val_batches
        best_vmae = running_vmae / num_val_batches

        val_curve.append(best_vloss)
        val_mae_curve.append(best_vmae)

        many_mae = best_buckets['mae_many']
        med_mae = best_buckets['mae_medium']
        few_mae = best_buckets['mae_few']

        bucket_many.append(many_mae)
        bucket_medium.append(med_mae)
        bucket_few.append(few_mae)

        current_lr = optimizer.param_groups[0]['lr']
        learning_rate_curve.append(current_lr)
        if scheduler is not None:
            scheduler.step()

        # Logging
        if many_mae <= CONFIG['MANY_SHOT_SAFETY_MARGIN']:
            constraint_symbol = "✅"  # Passes safety margin
        elif many_mae <= CONFIG['MANY_SHOT_CONSTRAINT']:
            constraint_symbol = "🎯"  # Passes constraint but risky
        else:
            constraint_symbol = ""

        print(f"Epoch {epoch+1:3d}/{CONFIG['NUM_EPOCHS']} | "
            f"TrL: {avg_train_loss:.5f} | VaL: {best_vloss:.5f} | "
            f"TrMAE: {avg_train_mae:.5f} | VaMAE: {best_vmae:.5f} | "
            f"Many: {many_mae:.4f}{constraint_symbol} | Med: {med_mae:.4f} | Few: {few_mae:.4f} | "
            f"LR: {current_lr:.6f}", end="")

        # Checkpointing - Priority: 1) Pass safety margin, 2) Minimize few-shot
        passes_safety_margin = many_mae <= CONFIG['MANY_SHOT_SAFETY_MARGIN']
        passes_constraint = many_mae <= CONFIG['MANY_SHOT_CONSTRAINT']

        if best_epoch == -1:
            improved = True
        elif passes_safety_margin:
            # Check if previous best also passed safety margin
            prev_passes_safety = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_SAFETY_MARGIN']
            if not prev_passes_safety:
                # First time passing safety margin - always save!
                improved = True
            else:
                # Both pass safety margin - choose lowest FEW-shot MAE (challenge objective!)
                improved = few_mae < bucket_few[best_epoch]
        elif passes_constraint:
            # Passes constraint but not safety margin - only save if better than previous
            prev_passes_constraint = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_CONSTRAINT']
            prev_passes_safety = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_SAFETY_MARGIN']
            if not prev_passes_constraint:
                # First time passing constraint - save it
                improved = True
            elif not prev_passes_safety:
                # Both pass constraint but not safety - choose lowest many-shot
                improved = many_mae < bucket_many[best_epoch]
            else:
                # Previous passes safety but current doesn't - don't save
                improved = False
        else:
            # Doesn't pass constraint - only improve if getting closer to constraint
            prev_passes_constraint = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_CONSTRAINT']
            improved = (not prev_passes_constraint) and (many_mae < bucket_many[best_epoch])

        if improved:
            best_loss = best_vloss
            best_epoch = epoch
            epochs_no_improve = 0

            checkpoint_dict = {
                'epoch': epoch,
                'model_type': CONFIG['MODEL_TYPE'],
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': best_vloss,
                'train_mae': avg_train_mae,
                'val_mae': best_vmae,
                'bucket_stats': best_buckets,
                'many_mae': many_mae,
                'few_mae': few_mae,
                'config': CONFIG
            }
            if scheduler is not None:
                checkpoint_dict['scheduler_state_dict'] = scheduler.state_dict()
            torch.save(checkpoint_dict, CONFIG['CHECKPOINT_NAME'])

            if passes_safety_margin:
                print(f" 💎 BEST - SAFE MARGIN! (Many≤{CONFIG['MANY_SHOT_SAFETY_MARGIN']}°)")
            elif passes_constraint:
                print(f" 🎯 BEST - PASSES CONSTRAINT (Risky: {many_mae:.4f}° → Kaggle ~{many_mae+0.05:.4f}°)")
            else:
                print(f" ⚠️  BEST (No constraint pass yet)")
        else:
            epochs_no_improve += 1
            if epochs_no_improve > CONFIG['EARLY_STOP_PATIENCE']:
                print(f"\n\nEarly stopping at epoch {epoch+1}")
                break
            else:
                print("")

    print(f"\nTraining complete")
    print(f"Best Epoch: {best_epoch+1}/{len(train_curve)}")
    print(f"Best Val Loss: {best_loss:.5f}")
    if best_epoch >= 0:
        print(f"Bucket MAE: Many={bucket_many[best_epoch]:.5f} | "
            f"Med={bucket_medium[best_epoch]:.5f} | Few={bucket_few[best_epoch]:.5f}")
    # ---- Return fold metrics ----
    result = {
        'val_file': val_file,
        'best_epoch': best_epoch,
        'many': float(bucket_many[best_epoch]) if best_epoch >= 0 else np.nan,
        'med': float(bucket_medium[best_epoch]) if best_epoch >= 0 else np.nan,
        'few': float(bucket_few[best_epoch]) if best_epoch >= 0 else np.nan,
        'ckpt': cfg['CHECKPOINT_NAME'],
    }
    return result, pipeline # Return pipeline here

In [ ]:
files = list_train_files(CONFIG['TRAIN_PATH'])
folds = make_file_folds(files, k=5, seed=CONFIG['SEED'], shuffle=True)

# Pick validation files for K=5 ensuring diversity
manual_val_files = [
    "train_part_017.csv",  # few-shot heavy
    "train_part_020.csv",  # balanced
    "train_part_010.csv",  # medium-heavy
    "train_part_012.csv",  # many-dominant (simple)
    "train_part_006.csv"   # balanced
]

fold_results = []
last_pipeline = None # Initialize pipeline variable
for val_file in manual_val_files:
    result, pipeline = run_one_fold(CONFIG, feature_names, val_file) # Capture pipeline
    fold_results.append(result)
    last_pipeline = pipeline # Store the pipeline from the last fold

print("\n===== K-FOLD SUMMARY =====")
for r in fold_results:
    print(f"{r['val_file']:<20s} | Many={r['many']:.4f} | Med={r['med']:.4f} | Few={r['few']:.4f}")

mean_many = np.mean([r['many'] for r in fold_results])
mean_med = np.mean([r['med'] for r in fold_results])
mean_few = np.mean([r['few'] for r in fold_results])
print(f"\n📊 Average across folds → Many={mean_many:.4f} | Med={mean_med:.4f} | Few={mean_few:.4f}")


🧩 Starting fold with validation file: train_part_017.csv

🎯 V2 Data Loader: Dedicated Validation File Strategy
Validation file: train_part_017.csv

🔵 train_part_001.csv        n=24,633 → TRAINING
🔵 train_part_002.csv        n=   615 → TRAINING
🔵 train_part_003.csv        n=32,215 → TRAINING
🔵 train_part_004.csv        n=10,249 → TRAINING
🔵 train_part_005.csv        n= 6,480 → TRAINING
🔵 train_part_006.csv        n=24,528 → TRAINING
🔵 train_part_007.csv        n=10,849 → TRAINING
🔵 train_part_008.csv        n=26,351 → TRAINING
🔵 train_part_009.csv        n=33,676 → TRAINING
🔵 train_part_010.csv        n=11,364 → TRAINING
🔵 train_part_011.csv        n=33,732 → TRAINING
🔵 train_part_012.csv        n= 7,292 → TRAINING
🔵 train_part_013.csv        n=10,701 → TRAINING
🔵 train_part_014.csv        n= 4,515 → TRAINING
🔵 train_part_015.csv        n= 9,685 → TRAINING
🔵 train_part_016.csv        n= 4,203 → TRAINING
✅ train_part_017.csv        n= 1,020 → VALIDATION
🔵 train_part_018.csv        n= 5,

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(



Training batches: 267
Validation batches: 2
Train samples: 136,559
Val samples: 521

📊 Standardization Statistics (from V2):
   Features standardized: 22
   Example - vxCG: mean=27.1860, std=12.8130
   Example - ayCG: mean=0.2027, std=5.6488

✅ Including validation samples in LDS bin statistics...
   Training samples: 136,559
   Validation samples: 521
   Total for LDS: 137,080
Device: cpu

📊 Input shape: (batch=512, seq=20, features=22)

[EnhancedLDS+FocalR Loss - CSV-Aligned Configuration]
Loaded 234 band definitions from CSV
Approximate boundaries (max |sideslip| per band):
  Many-shot:   <= 2.4000 degrees
  Medium-shot: <= 5.0000 degrees
Data distribution:
  Many:   124956 (91.16%)
  Medium:   9521 ( 6.95%)
  Few:      2603 ( 1.90%)
Bucket weights: 1.0 / 2.0 / 4.0
Focal gamma: 1.0 | Huber beta: 0.08
LDS: kernel=gaussian, sigma=2.5


🔧 Model: GRU_Fixed
   Total parameters: 1,103,405
   Trainable parameters: 1,103,405
Training setup complete
Epoch   1/100 | TrL: 1.87327 | VaL: 1.019

KeyboardInterrupt: 

In [ ]:
import torch
import numpy as np
import pandas as pd

# --- Load test data ---
print("Loading test data for final ensemble prediction...")
test_dataloader = load_test_data(
    test_path=CONFIG['TEST_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=last_pipeline,  # use the same preprocessing pipeline
    seed=CONFIG['SEED'],
    load_anonymized_data=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Ensemble Predictions from all fold checkpoints ---
fold_checkpoints = [r['ckpt'] for r in fold_results if r and 'ckpt' in r]

print(f"\nFound {len(fold_checkpoints)} fold checkpoints:")
for ckpt in fold_checkpoints:
    print("  -", ckpt)

all_fold_preds = []

for ckpt_path in fold_checkpoints:
    print(f"\n🔹 Loading checkpoint: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)

    model = GRUModel(
        input_size=CONFIG['INPUT_SIZE'],
        output_size=1,
        rnn_units=CONFIG['RNN_UNITS'],
        fc_units=CONFIG['FC_UNITS'],
        dropout=CONFIG['DROPOUT'],
        num_layers=CONFIG['NUM_LAYERS'],
        use_batch_norm=CONFIG['USE_BATCH_NORM']
    ).to(device)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Generate predictions
    preds_fold = []
    with torch.no_grad():
        for xb, _ in test_dataloader:
            xb = xb.to(device)
            pred = model(xb)
            preds_fold.append(pred.cpu().numpy())

    preds_fold = np.concatenate(preds_fold).flatten()
    print(f"  ↳ Fold predictions shape: {preds_fold.shape}, mean={preds_fold.mean():.4f}")
    all_fold_preds.append(preds_fold)

# --- Average across folds ---
test_preds_mean = np.mean(all_fold_preds, axis=0)
print(f"\n✅ Averaged predictions shape: {test_preds_mean.shape}")
print(f"Range: [{test_preds_mean.min():.3f}°, {test_preds_mean.max():.3f}°]")
print(f"Mean: {test_preds_mean.mean():.3f}° | Std: {test_preds_mean.std():.3f}°")

# --- Sanity check ---
if len(test_preds_mean) == 49328:
    print("✅ Correct number of predictions (49,328).")
else:
    print(f"⚠️ Unexpected length: {len(test_preds_mean)}")

# --- Save to CSV ---
submission_path = CONFIG['SUBMISSION_CSV']
y_pred = pd.DataFrame({
    "id": range(len(test_preds_mean)),
    "predicted_sideslip": test_preds_mean
})
y_pred.to_csv(submission_path, index=False)

print(f"\n📁 Saved ensemble predictions to: {submission_path}")
print("🎯 Ready for Kaggle submission!")

Loading test data for final ensemble prediction...


NameError: name 'pipeline' is not defined